In [17]:
import torch
from transformers import PeAudioVideoModel, PeAudioVideoProcessor,AutoProcessor, AutoModel
import time
import numpy as np
from pathlib import Path
import glob
import os
import re
import torch
from natsort import natsorted  
import gc

# Base paths
BASE_DIR = Path("/home/amin/Research/Representation/Movie/data/Setareh/Data")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device is {DEVICE}")

device is cuda


In [2]:
model_id = "facebook/pe-av-small-16-frame"
import os
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"  # 5 minutes

model = PeAudioVideoModel.from_pretrained(model_id).to(DEVICE)
processor = PeAudioVideoProcessor.from_pretrained(model_id)
print("Model & Processor Ready")

[ERROR] `padding_mask_videos` is part of PeVideoModel.__init__.<locals>.get_video_features's signature, but not documented. Make sure to add it to the docstring of the function in /home/amin/miniconda3/envs/peav/lib/python3.11/site-packages/transformers/models/pe_video/modeling_pe_video.py.


Loading weights:   0%|          | 0/859 [00:00<?, ?it/s]

Model & Processor Ready


# QC

In [ ]:
# --- CONFIG ---
SEGMENT_DURATIONS = [1.0, 2.0, 5.0, 10.0]  
# ----------------

def extract_part_number(filename):
    match = re.search(r'part[_-]?(\d+)', filename.stem, re.IGNORECASE)
    if match:
        return int(match.group(1))
    return None

print(f"Checking data compatibility in: {BASE_DIR}\n")

for seg_sec in SEGMENT_DURATIONS:
    all_videos = []
    all_audios = []
    
    # Optional: Logic for sliding window steps could go here
    # steps = [seg_sec] 
    # for step in steps: 

    print(f"--- Checking {int(seg_sec)}s segments ---")

    for run_idx in range(1, 19):
        vid_chunk_dir = BASE_DIR / f"Video{run_idx}" / f"Video{run_idx}_chunks_{int(seg_sec)}s"
        aud_chunk_dir = BASE_DIR / f"Audio{run_idx}" / f"Audio{run_idx}_chunks_{int(seg_sec)}s"

        if not vid_chunk_dir.exists() or not aud_chunk_dir.exists():
            print(f"Video {run_idx}: 0 paired chunks (Dir missing)")
            continue

        vids = natsorted(list(vid_chunk_dir.glob("*.mp4")))
        # Search for .wav (preferred) or fallback to .m4a if needed
        auds = natsorted(list(aud_chunk_dir.glob("*.wav")))
        if not auds: 
             auds = natsorted(list(aud_chunk_dir.glob("*.m4a")))

        vid_parts = {extract_part_number(v): v for v in vids if extract_part_number(v) is not None}
        aud_parts = {extract_part_number(a): a for a in auds if extract_part_number(a) is not None}

        common_parts = sorted(set(vid_parts.keys()) & set(aud_parts.keys()))

        for part_num in common_parts:
            all_videos.append(vid_parts[part_num])
            all_audios.append(aud_parts[part_num])

        if len(vids) == len(auds):    
            print(f"Video {run_idx}: {len(common_parts)} paired chunks (video: {len(vids)}, audio: {len(auds)})")
        else:
            raise RuntimeError("ERROR! mismatched segment length")

    paired = list(zip(all_videos, all_audios))
    print(f"{'='*60}")
    print(f"Total: {len(paired)} paired {int(seg_sec)}s segments")
    print(f"{'='*60}\n")

Checking data compatibility in: /home/amin/Research/Representation/Movie/data/Setareh/Data

--- Checking 1s segments ---
Video 1: 212 paired chunks (video: 212, audio: 212)
Video 2: 220 paired chunks (video: 220, audio: 220)
Video 3: 186 paired chunks (video: 186, audio: 186)
Video 4: 62 paired chunks (video: 62, audio: 62)
Video 5: 82 paired chunks (video: 82, audio: 82)
Video 6: 226 paired chunks (video: 226, audio: 226)
Video 7: 258 paired chunks (video: 258, audio: 258)
Video 8: 248 paired chunks (video: 248, audio: 248)
Video 9: 82 paired chunks (video: 82, audio: 82)
Video 10: 180 paired chunks (video: 180, audio: 180)
Video 11: 184 paired chunks (video: 184, audio: 184)
Video 12: 202 paired chunks (video: 202, audio: 202)
Video 13: 140 paired chunks (video: 140, audio: 140)
Video 14: 82 paired chunks (video: 82, audio: 82)
Video 15: 232 paired chunks (video: 232, audio: 232)
Video 16: 230 paired chunks (video: 230, audio: 230)
Video 17: 254 paired chunks (video: 254, audio: 254)

# Embedding extraction and saving megaloop

In [19]:
import torch
import numpy as np
import os
import re
from pathlib import Path
from natsort import natsorted

# --- CONFIG ---
OUTPUT_BASE = Path("/home/amin/Research/Representation/Movie/outputs/model_embeddings/pe-av-small-16-frame")
BATCH_SIZE = 8
SEGMENT_DURATIONS = [1.0, 2.0, 5.0, 10.0] 
# ----------------

for seg_sec in SEGMENT_DURATIONS:
    
    # 1. Gather Pairs (Re-running logic to ensure safety per loop iteration)
    all_videos = []
    all_audios = []
    
    for run_idx in range(1, 19):
        vid_chunk_dir = BASE_DIR / f"Video{run_idx}" / f"Video{run_idx}_chunks_{int(seg_sec)}s"
        aud_chunk_dir = BASE_DIR / f"Audio{run_idx}" / f"Audio{run_idx}_chunks_{int(seg_sec)}s"
        
        if not vid_chunk_dir.exists() or not aud_chunk_dir.exists():
            continue

        vids = natsorted(list(vid_chunk_dir.glob("*.mp4")))
        auds = natsorted(list(aud_chunk_dir.glob("*.wav")))
        if not auds: auds = natsorted(list(aud_chunk_dir.glob("*.m4a")))

        vid_parts = {extract_part_number(v): v for v in vids if extract_part_number(v) is not None}
        aud_parts = {extract_part_number(a): a for a in auds if extract_part_number(a) is not None}
        common_parts = sorted(set(vid_parts.keys()) & set(aud_parts.keys()))

        for part_num in common_parts:
            all_videos.append(vid_parts[part_num])
            all_audios.append(aud_parts[part_num])
        if len(vids) == len(auds):    
            print(f"Video {run_idx}: {len(common_parts)} paired chunks (video: {len(vids)}, audio: {len(auds)})")
        else:
            raise RuntimeError("ERROR! mismatched segment length")

    paired = list(zip(all_videos, all_audios))
    print(f"{'='*60}")
    print(f"Total: {len(paired)} paired {int(seg_sec)}s segments")
    print(f"{'='*60}")

    if len(paired) == 0:
        continue

    # 2. Extract
    OUTPUT_PATH = OUTPUT_BASE / f"{int(seg_sec)}s"
    os.makedirs(OUTPUT_PATH, exist_ok=True)
    
    video_out, audio_out, av_out = [], [], []

    with torch.inference_mode():
        for i in range(0, len(paired), BATCH_SIZE):
            batch_pairs = paired[i:i + BATCH_SIZE]
            v_batch = [str(v) for v, a in batch_pairs]
            a_batch = [str(a) for v, a in batch_pairs]
            
            inputs = processor(videos=v_batch, audio=a_batch, return_tensors="pt").to(torch.float32).to(DEVICE)
            
            with torch.autocast(device_type="cuda", dtype=torch.float32):
                outputs = model(**inputs)
            
            video_out.append(outputs.video_embeds.float().cpu())
            audio_out.append(outputs.audio_embeds.float().cpu())
            av_out.append(outputs.audio_video_embeds.float().cpu())
            
            if (i // BATCH_SIZE) % 50 == 0 or i == 0:
                print(f"Processed {i + len(batch_pairs)}/{len(paired)}")

    # 3. Save
    video_embeds = torch.cat(video_out, dim=0)
    audio_embeds = torch.cat(audio_out, dim=0)
    av_embeds = torch.cat(av_out, dim=0)

    print(f"Done!")
    print(f"Video embeddings: {video_embeds.shape}")
    print(f"Audio embeddings: {audio_embeds.shape}")
    print(f"AV embeddings:    {av_embeds.shape}")

    np.save(OUTPUT_PATH / 'peav_small_16-frame_av.npy', av_embeds.numpy())
    np.save(OUTPUT_PATH / 'peav_small_16-frame_a.npy', audio_embeds.numpy())
    np.save(OUTPUT_PATH / 'peav_small_16-frame_v.npy', video_embeds.numpy())
    print(f"embedding npy files saved to {OUTPUT_PATH}\n")
    del av_embeds,audio_embeds, video_embeds
    gc.collect()

Model & Processor Ready
Video 1: 212 paired chunks (video: 212, audio: 212)
Video 2: 220 paired chunks (video: 220, audio: 220)
Video 3: 186 paired chunks (video: 186, audio: 186)
Video 4: 62 paired chunks (video: 62, audio: 62)
Video 5: 82 paired chunks (video: 82, audio: 82)
Video 6: 226 paired chunks (video: 226, audio: 226)
Video 7: 258 paired chunks (video: 258, audio: 258)
Video 8: 248 paired chunks (video: 248, audio: 248)
Video 9: 82 paired chunks (video: 82, audio: 82)
Video 10: 180 paired chunks (video: 180, audio: 180)
Video 11: 184 paired chunks (video: 184, audio: 184)
Video 12: 202 paired chunks (video: 202, audio: 202)
Video 13: 140 paired chunks (video: 140, audio: 140)
Video 14: 82 paired chunks (video: 82, audio: 82)
Video 15: 232 paired chunks (video: 232, audio: 232)
Video 16: 230 paired chunks (video: 230, audio: 230)
Video 17: 254 paired chunks (video: 254, audio: 254)
Video 18: 82 paired chunks (video: 82, audio: 82)
Total: 3162 paired 1s segments
Processed 8/316

# Misc

### residual embeddings

In [13]:
import numpy as np
import statsmodels.api as sm
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
audio_emb = np.load(Path('/home/amin/Research/Representation/Movie/outputs/model_embeddings/pe-av-small-16-frame/peav_small_16frame_a.npy'))
video_emb = np.load(Path('/home/amin/Research/Representation/Movie/outputs/model_embeddings/pe-av-small-16-frame/peav_small_16frame_v.npy'))
joint_emb = np.load(Path('/home/amin/Research/Representation/Movie/outputs/model_embeddings/pe-av-small-16-frame/peav_small_16frame_av.npy'))

In [30]:
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt

# Assuming you have:
# audio_emb: (1581, 1024)
# video_emb: (1581, 1024)
# joint_emb: (1581, 1024)

# Concatenate audio and video as predictors
X = np.concatenate([audio_emb, video_emb], axis=1)  # (1581, 2048)

# Fit multivariate OLS - one model for all dimensions at once
model = sm.OLS(joint_emb, X, hasconst=False)
results = model.fit()

# Get predictions and residuals
predictions = results.fittedvalues  # (1581, 1024)
residuals = results.resid  # (1581, 1024) - THIS IS YOUR UNIQUE JOINT INFO

residuals_normalized_l2 = residuals / (np.linalg.norm(residuals, axis=1, keepdims=True) + 1e-10)
residuals_normalized_zscore = (residuals - residuals.mean(axis=0)) / (residuals.std(axis=0) + 1e-10)

np.save(Path('/home/amin/Research/Representation/Movie/outputs/model_embeddings/pe-av-small-16-frame/peav_small_16frame_av_ols_residual.npy'), 
        residuals_normalized_l2)

In [ ]:
from sklearn.decomposition import PCA

# Get shared space
X_combined = np.concatenate([audio_emb, video_emb], axis=1)
pca = PCA(n_components=1024)
pca.fit(X_combined)

# Project joint embedding
joint_in_shared_space = pca.transform(joint_emb)
joint_reconstructed = pca.inverse_transform(joint_in_shared_space)

# Orthogonal component
unique_joint = joint_emb - joint_reconstructed

# Variance analysis
shared_variance = np.var(joint_reconstructed, axis=0).sum()
unique_variance = np.var(unique_joint, axis=0).sum()
total_variance = np.var(joint_emb, axis=0).sum()

np.float32(0.014503118)

In [ ]:
from sklearn.cross_decomposition import CCA

n_components = min(512, 1024)  # Choose based on your needs
cca = CCA(n_components=n_components)

# Fit on audio vs joint
audio_c, joint_c = cca.fit_transform(audio_emb, joint_emb)

# Measure correlation per component
correlations = [np.corrcoef(audio_c[:, i], joint_c[:, i])[0, 1] 
                for i in range(n_components)]

# Low correlation components indicate unique joint information

In [ ]:
X = np.concatenate([audio_emb, video_emb], axis=1)
Q, R = np.linalg.qr(X)
projection = Q @ (Q.T @ joint_emb)
orthogonal = joint_emb - projection

np.save(Path('/home/amin/Research/Representation/Movie/outputs/model_embeddings/pe-av-small-16-frame/peav_small_16frame_av_projection_residual.npy'), 
        orthogonal)

(1581, 1024)